# Persian Text Summarization Dataset Pipeline

in this notebook  i implement a sample pipeline for creating a Persian text summarization dataset. The main steps are:

1. **Data Acquisition**: Load pn-summery dataset of summerized Persian articles, select a subset of size 52, and split into  48 training/ 2 dev/ 2 test sets.
2. **Data Cleaning & Organization**: Hazm and pandas to normalize and tokenize the Persian text. although they are already normalized in this dataset
3. **Clustering**: Convert texts to embeddings using a Persian embedding model and group similar texts using DBSCAN/HDBSCAN.
4. **Summarization**: Generate document-level abstractive summaries using Persian-capable MT5 model with one-shot learning.
5. **Evaluation**: Evaluate the generated summaries using ROUGE metrics.

- first install everything needed with ( pip install -r requirements.txt ).

In [7]:
import pandas as pd
import numpy as np

dev = pd.read_csv('pn_summary_dev.csv', sep='\t')
dev["article"] = dev["article"].apply(lambda t: t.replace("[n]", "\n"))
dev["summary"] = dev["summary"].apply(lambda t: t.replace("[n]", "\n"))
print("Original dataset shape:", dev.shape)  # expected shape is (5592, 8)

sample_df = dev.sample(n=50, random_state=42).reset_index(drop=True)
print("Sampled dataset shape:", sample_df.shape)

# 48 for train, 2 for dev, 2 for test
train_df = sample_df.iloc[:48].reset_index(drop=True)
dev_df = sample_df.iloc[48:50].reset_index(drop=True)
test_df = sample_df.iloc[50:52].reset_index(drop=True)  

if len(sample_df) < 52:
    dev_df = sample_df.iloc[-4:-2].reset_index(drop=True)
    test_df = sample_df.iloc[-2:].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Test shape:", test_df.shape)

Original dataset shape: (5592, 8)
Sampled dataset shape: (50, 8)
Train shape: (48, 8)
Dev shape: (2, 8)
Test shape: (2, 8)


## Step 2: Data Cleaning & Organization

for now i'm using **Hazm** to normalize and tokenize the articles but it's not necessary as they are already normalized.

In [8]:
from hazm import Normalizer, word_tokenize
normalizer = Normalizer()

def clean_text(text):
    text = normalizer.normalize(text)
    tokens = word_tokenize(text)
    return ' '.join(tokens)

train_df['clean_article'] = train_df['article'].apply(clean_text)
print(train_df[['article', 'clean_article']].head())

                                             article  \
0  به گزارش ایمنا، محمدعلی جاوید اظهار کرد: پیش ا...   
1  به گزارش روز چهارشنبه ایرنا از پایگاه اطلاع‌رس...   
2  به گزارش شانا، سید مصطفی موسوی، مدیرعامل شرکت ...   
3  قاسم میرزایی نیکو، نماینده مجلس دهم، در رابطه ...   
4  به گزارش شانا رییس واحد ابزار دقیق خطوط لوله و...   

                                       clean_article  
0  به گزارش ایمنا ، محمدعلی جاوید اظهار کرد : پیش...  
1  به گزارش روز چهارشنبه ایرنا از پایگاه اطلاع‌رس...  
2  به گزارش شانا ، سید مصطفی موسوی ، مدیرعامل شرک...  
3  قاسم میرزایی نیکو ، نماینده مجلس دهم ، در رابط...  
4  به گزارش شانا رییس واحد ابزار دقیق خطوط لوله و...  


## Step 3: Clustering

in this step i convert the cleaned texts into embeddings using Persian Sentence transformer model. then i will apply a density based clustering algorithm ( HDBSCAN) to group similar texts(later we can test dbscan but it requires a lot of hyperparameter tuning). 

In [9]:
import os  
import numpy as np  
import torch  
from sentence_transformers import SentenceTransformer  
import hdbscan  
from sklearn.preprocessing import normalize  

# tested a few embedding models and this one was better in my opinion  
embed_model = SentenceTransformer('heydariAI/persian-embeddings')  

def get_embedding(text):  
    embedding = embed_model.encode(text, convert_to_tensor=True)  
    return embedding.cpu().detach().numpy()  

# get embeddings for the cleaned articles in training set  
train_texts = train_df['clean_article'].tolist()  
embeddings = np.vstack([get_embedding(text) for text in train_texts])  
print("Embeddings shape:", embeddings.shape)  

# Normalize embeddings to unit vectors so we can use Euclidean distance instead of cosine cause i got an error using cosine
#it's actually similar to cosine after normalization
normalized_embeddings = normalize(embeddings, norm='l2')

# apply HDBSCAN clustering using Euclidean metric on normalized vectors 
clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')  
cluster_labels = clusterer.fit_predict(normalized_embeddings)   

# add cluster labels to the train_df  
train_df['cluster'] = cluster_labels  
print(train_df[['clean_article', 'cluster']].head())  

# test to see
unique_clusters = [c for c in np.unique(cluster_labels) if c != -1]  
print(f"Number of clusters found (excluding noise): {len(unique_clusters)}")  



Embeddings shape: (48, 1024)
                                       clean_article  cluster
0  به گزارش ایمنا ، محمدعلی جاوید اظهار کرد : پیش...       -1
1  به گزارش روز چهارشنبه ایرنا از پایگاه اطلاع‌رس...       -1
2  به گزارش شانا ، سید مصطفی موسوی ، مدیرعامل شرک...        1
3  قاسم میرزایی نیکو ، نماینده مجلس دهم ، در رابط...        1
4  به گزارش شانا رییس واحد ابزار دقیق خطوط لوله و...        1
Number of clusters found (excluding noise): 3


c:\play\programming\python\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\play\programming\python\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


## Step 4: Summarization

We now generate document-level abstractive summaries using a MT5 model specialized in summerization with 2 persian summery datasets. We define a function `generate_summary` to create summaries for each document. 

additionally pick one representative summary from each cluster to be used later for one-shot learning (later we can use few-shot learning)

In [10]:
from transformers import AutoModelForSeq2SeqLM, MT5Tokenizer

sum_model = AutoModelForSeq2SeqLM.from_pretrained('nafisehNik/mt5-persian-summary')
tokenizer = MT5Tokenizer.from_pretrained('nafisehNik/mt5-persian-summary')

def generate_summary(model, abstract, num_beams=15, repetition_penalty=0.8,
                     length_penalty=1.0, early_stopping=True, max_output_length=140):
    # Encode input text
    source_encoding = tokenizer(abstract, max_length=1000, padding="max_length", truncation=True,
                                return_attention_mask=True, add_special_tokens=True, return_tensors="pt")
    
    # Generate summary IDs
    generated_ids = model.generate(
        input_ids=source_encoding["input_ids"],
        attention_mask=source_encoding["attention_mask"],
        num_beams=num_beams,
        max_length=max_output_length,
        repetition_penalty=repetition_penalty,
        length_penalty=length_penalty,
        early_stopping=early_stopping,
        use_cache=True
    )
    
    # Decode generated IDs
    preds = [tokenizer.decode(gen_id, skip_special_tokens=True, clean_up_tokenization_spaces=True) 
             for gen_id in generated_ids]
    return " ".join(preds)

#example
print("--- Example Summarization ---")
example_text = train_df.loc[0, 'clean_article']
print("Original Text (truncated):", example_text[:200], "...\n")
example_generated_summary = generate_summary(sum_model, abstract=example_text, num_beams=15, max_output_length=140)
print("Generated Summary:", example_generated_summary)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.


--- Example Summarization ---
Original Text (truncated): به گزارش ایمنا ، محمدعلی جاوید اظهار کرد : پیش از این به ظرفیت واقعی شهرداری از نظر بودجه نرسیده_بودیم طوری که بودجه شهرداری در دو سال قبل ۶۰ میلیارد تومان بود که اگر ۱۰۰ درصد تحقق پیدا می‌کرد می‌توان ...

Generated Summary: ، گفت: در حال حاضر حدود ۷۰ میلیارد تومان هزینه حقوق و مزایا و هزینه های جاری شهرداری است.


## Step 5: create cluster_summaries

This cell iterates through each non-noise cluster, calculates the cluster centroid (with the embeddings i got above), and determines the document (its index) that is closest to that centroid using cosine similarity. It then retrieves that document's cleaned text and its original (gold) summary from train_df and it generates a new summary using the MT5 model and stores both the gold and generated summaries in a dictionary called cluster_summaries.

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def get_representative_index(embeds, indices):
    """
    Given a set of indices for a cluster, compute the centroid of embeddings and return the index
    that is closest to the centroid.
    """
    cluster_embeds = embeds[indices]
    centroid = np.mean(cluster_embeds, axis=0).reshape(1, -1)
    sims = cosine_similarity(cluster_embeds, centroid)
    rep_local_index = np.argmax(sims)
    return indices[rep_local_index]

cluster_summaries = {}

unique_clusters = [c for c in np.unique(train_df['cluster']) if c != -1]

for cluster in unique_clusters:
    # find indices for documents in the current cluster
    cluster_indices = train_df.index[train_df['cluster'] == cluster].tolist()
    # select the representative index based on cosine similarity to the cluster centroid
    rep_index = get_representative_index(embeddings, cluster_indices)
    
    # get the representative document's cleaned text and gold summary
    rep_text = train_df.loc[rep_index, 'clean_article']
    rep_gold_summary = train_df.loc[rep_index, 'summary']
    
    # generate a summary using the persian fine-tuned MT5 model for this document
    rep_generated_summary = generate_summary(sum_model, abstract=rep_text, num_beams=15, max_output_length=140)
    
    # store both summaries in the dictionary
    cluster_summaries[cluster] = {
        "generated_summary": rep_generated_summary,
        "gold_summary": rep_gold_summary
    }

print("Defined cluster_summaries:")
print(cluster_summaries)


Defined cluster_summaries:
{0: {'generated_summary': 'گزارش وضعیت املاک شهرداری یکی از موضوعات و دغدغه های اصلی اعضای شورای شهر است.', 'gold_summary': 'رئیس کمیسیون فرهنگی شورای اسلامی شهر تهران گفت: لازم است شهردار تهران، بلافاصله نسبت به ارائه گزارش در خصوص آخرین وضعیت املاک شهرداری و اقدامات انجام شده برای ساماندهی آن\u200cها اقدام کند.'}, 1: {'generated_summary': 'نشان می دهد که توافق هسته ای ایران می تواند فرصت بی همتایی را برای شرکتهای نفت خارجی در ایران فراهم کند.', 'gold_summary': 'کارشناسان نفتی و اقتصاددانان معتقدند توافق هسته\u200cای ایران گامی مثبت در جهت برداشتن تحریمها علیه ایران و در نتیجه ورود دوباره شرکتهای معتبر نفتی به صنعت نفت ایران است.'}, 2: {'generated_summary': '، گفت: در هفته گذشته ۴۵ حادثه پوشش امدادی داده شده است و ۱۳۷ آسیب دیده ناشی از حوادث در این مدت از خدمات امدادی هلال احمر استان اصفهان بهره مند شدند.', 'gold_summary': 'معاون امداد و نجات جمعیت هلال\u200cاحمر استان اصفهان از امدادرسانی به ۱۳۷ آسیب دیده ناشی از ۴۵ حادثه به وسیله ۴۳ تیم عملیاتی این جمعیت د

## Step 6: Evaluation

this is a special rouge made by pn-summary that is capable of evaluating persian text as the original rouge does not support persian.

In [12]:
from rouge_score import rouge_scorer
from prettytable import PrettyTable

def evaluate_summary_persian(reference, prediction):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'],use_stemmer=False) 
    return scorer.score(reference, prediction)

print("\nEvaluation of Representative Summaries per Cluster:")
table = PrettyTable(title='ROUGE Scores for Clusters')
table.field_names = ["Cluster", "ROUGE", "Precision", "Recall", "F1"]

for cl, summ in cluster_summaries.items():
    gold = summ['gold_summary']
    generated = summ['generated_summary']

    scores = evaluate_summary_persian(gold, generated)

    for rouge_type, score in scores.items():
        table.add_row([cl, rouge_type,
                       f"{score.precision*100:.2f}",
                       f"{score.recall*100:.2f}",
                       f"{score.fmeasure*100:.2f}"])
    table.add_row([''] * 5)
    table.add_row(['***'] * 5)
    table.add_row([''] * 5)

print(table)




Evaluation of Representative Summaries per Cluster:
+-----------------------------------------------+
|           ROUGE Scores for Clusters           |
+---------+--------+-----------+--------+-------+
| Cluster | ROUGE  | Precision | Recall |   F1  |
+---------+--------+-----------+--------+-------+
|    0    | rouge1 |   53.33   | 24.24  | 33.33 |
|    0    | rougeL |   33.33   | 15.15  | 20.83 |
|         |        |           |        |       |
|   ***   |  ***   |    ***    |  ***   |  ***  |
|         |        |           |        |       |
|    1    | rouge1 |   36.36   | 26.67  | 30.77 |
|    1    | rougeL |   31.82   | 23.33  | 26.92 |
|         |        |           |        |       |
|   ***   |  ***   |    ***    |  ***   |  ***  |
|         |        |           |        |       |
|    2    | rouge1 |   51.61   | 50.00  | 50.79 |
|    2    | rougeL |   22.58   | 21.88  | 22.22 |
|         |        |           |        |       |
|   ***   |  ***   |    ***    |  ***   |  *** 

## Conclusion

conclusion is that i'm both capable and enthusiastic about this project  😁
thank you for reading ❤️